In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS bronze;
CREATE SCHEMA IF NOT EXISTS silver;
CREATE SCHEMA IF NOT EXISTS gold;


In [0]:
import pyspark
import pyspark.sql as sql
from pyspark.sql import functions as sf
from pyspark.sql import SparkSession
import os
from pyspark.sql.functions import current_timestamp

path = "/Volumes/workspace/default/ocorrencias-aeronautica/data/"

tabelas_csv = {
    "aeronave.csv": "cenipa_aeronave",
    "fator_contribuinte.csv": "cenipa_contribuinte",
    "ocorrencia_tipo.csv": "cenipa_tipo",
    "ocorrencia.csv": "cenipa_ocorrencia",
    "recomendacao.csv": "cenipa_recomendacao"
}

for csv, tabela in tabelas_csv.items():
    filepath = f"{path}{csv}" # caminho completo para pegar o arquivo
    print(f"Ingerindo {csv} para a tabela {tabela}...")

    df_raw = spark.read.csv(filepath, header=True, inferSchema=True, sep=";", mode="PERMISSIVE", columnNameOfCorruptRecord=f'_corrupted_data_{tabela}')
    df_bronze = df_raw\
                .withColumn("_ingestion_date", current_timestamp())

    df_bronze.write.format("delta").mode("overwrite").option("MergeSchema", "true").saveAsTable(f"bronze.{tabela}")

In [0]:
%sql
DELETE FROM bronze.cenipa_aeronave 
WHERE aeronave_matricula IS NULL;

DELETE FROM bronze.cenipa_tipo
WHERE ocorrencia_tipo is null

In [0]:
%sql

-- ==========================================
-- 1. TABELA OCORRÊNCIA (Chave Simples)
-- ==========================================
-- Garantir que a coluna não seja nula
-- ALTER TABLE bronze.cenipa_ocorrencia ALTER COLUMN codigo_ocorrencia SET NOT NULL;
-- -- Criar a Constraint de PK
-- ALTER TABLE bronze.cenipa_ocorrencia ADD CONSTRAINT pk_ocorrencia PRIMARY KEY (codigo_ocorrencia);


-- ==========================================
-- 2. TABELA AERONAVE (Chave Composta)
-- ==========================================
-- ALTER TABLE bronze.cenipa_aeronave ALTER COLUMN codigo_ocorrencia2 SET NOT NULL;
-- ALTER TABLE bronze.cenipa_aeronave ALTER COLUMN aeronave_matricula SET NOT NULL;

-- ALTER TABLE bronze.cenipa_aeronave ADD CONSTRAINT pk_aeronave PRIMARY KEY (codigo_ocorrencia2, aeronave_matricula);


-- ==========================================
-- 3. TABELA FATOR CONTRIBUINTE (Chave Composta)
-- ==========================================
-- ALTER TABLE bronze.cenipa_contribuinte ALTER COLUMN codigo_ocorrencia3 SET NOT NULL;
-- ALTER TABLE bronze.cenipa_contribuinte ALTER COLUMN fator_nome SET NOT NULL;

-- ALTER TABLE bronze.cenipa_contribuinte ADD CONSTRAINT pk_contribuinte PRIMARY KEY (codigo_ocorrencia3, fator_nome);


-- ==========================================
-- 4. TABELA OCORRÊNCIA TIPO (Chave Composta)
-- ==========================================
ALTER TABLE bronze.cenipa_tipo ALTER COLUMN codigo_ocorrencia1 SET NOT NULL;
ALTER TABLE bronze.cenipa_tipo ALTER COLUMN ocorrencia_tipo SET NOT NULL;

ALTER TABLE bronze.cenipa_tipo ADD CONSTRAINT pk_tipo PRIMARY KEY (codigo_ocorrencia1, ocorrencia_tipo);


-- ==========================================
-- 5. TABELA RECOMENDAÇÃO (Chave Composta)
-- ==========================================
ALTER TABLE bronze.cenipa_recomendacao ALTER COLUMN codigo_ocorrencia4 SET NOT NULL;
ALTER TABLE bronze.cenipa_recomendacao ALTER COLUMN recomendacao_numero SET NOT NULL;

ALTER TABLE bronze.cenipa_recomendacao ADD CONSTRAINT pk_recomendacao PRIMARY KEY (codigo_ocorrencia4, recomendacao_numero);

In [0]:
%sql
